# Laboratório de Criptografia: Implementação e Quebra

Vinicius Trindade Dias Abel - UFMG

## Bibliotecas

In [1036]:
import unicodedata
import re
from collections import Counter, defaultdict
from itertools import product

## Tarefa 1: Implementação da Cifra de César

### Função de Normalização

In [1037]:
def normalize(text, keep_spaces=True):
    """
    Normaliza um texto para A-Z e espaços:
    - converte para maiúsculas
    - remove acentos 
    - remove caracteres que não sejam A-Z (e espaços, se keep_spaces=False)
    - colapsa múltiplos espaços e remove espaços nas pontas

    Parâmetros:
    - text: string de entrada
    - keep_spaces: se True, mantém espaços; se False, remove espaços
    
    Retorno:
    - string normalizada
    """
    # Se o texto for vazio, retorna string vazia
    if text is None:
        return ''

    # Garante que é string
    if not isinstance(text, str):
        text = str(text)

    # Converte para maiúsculas
    text = text.upper()

    # Remove acentos
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(char for char in text if not unicodedata.combining(char))

    # Mantém apenas A-Z (e espaços, se keep_spaces=True)
    if keep_spaces:
        text = re.sub(r'[^A-Z ]+', '', text)
    else:
        text = re.sub(r'[^A-Z]+', '', text)

    # Colapsa múltiplos espaços
    text = re.sub(r' +', ' ', text)
    
    # Remove espaços nas pontas
    return text.strip()

### Cifra de César - Cifrar

In [1038]:
def cesar_encrypt(text: str, k: int) -> str:
    """
    Parâmetros:
    - text: string de entrada (plaintext)
    - k: deslocamento (0-25)

    Retorno:
    - texto cifrado. Mantém espaços, apenas letras são deslocadas.
    """
    # Normaliza entrada
    text = normalize(text)

    # Cifra de César
    result = []
    for char in text:
        if char == " ":
            result.append(" ")
        else:
            displaced = (ord(char) - ord("A") + k) % 26
            result.append(chr(ord("A") + displaced))
    return "".join(result)

### Cifra de César - Decifrar

In [1039]:
def cesar_decrypt(text: str, k: int) -> str:
    """
    Parâmetros:
    - text: string de entrada (ciphertext)
    - k: deslocamento (0-25)

    Retorno:
    - texto decifrado. Mantém espaços, apenas letras são deslocadas.
    """
    # Normaliza entrada
    text = normalize(text)

    # Decifra de César
    result = []
    for char in text:
        if char == " ":
            result.append(" ")
        else:
            displaced = (ord(char) - ord("A") - k) % 26
            result.append(chr(ord("A") + displaced))
    return "".join(result)

### Validação

In [1040]:
msg = "SEGURANCA DA INFORMACAO E IMPORTANTE"
k = 3

encrypt = cesar_encrypt(msg, k)
decrypt = cesar_decrypt(encrypt, k)

print("Mensagem original:", msg)
print("Cifrado (k=3):", encrypt)
print("Decifrado:", decrypt)
print("Recuperou original?", decrypt == normalize(msg))

Mensagem original: SEGURANCA DA INFORMACAO E IMPORTANTE
Cifrado (k=3): VHJXUDQFD GD LQIRUPDFDR H LPSRUWDQWH
Decifrado: SEGURANCA DA INFORMACAO E IMPORTANTE
Recuperou original? True


### Resultado

Para validar a implementação, a mensagem `SEGURANCA DA INFORMACAO E IMPORTANTE` foi cifrada com `k = 3`. Obtendo como resultado `VHJXUDQFD GD LQIRUPDFDR H LPSRUWDQWH`.

Em seguida, o texto foi decifrado com a mesma chave `k = 3`, revelando novamente a mensagem original: `SEGURANCA DA INFORMACAO E IMPORTANTE`.

#### Caracteres não alfabéticos

Durante a normalização:
- **acentos** são removidos (ex.: `Ç → C`, `Á → A`);
- **números, pontuações e símbolos** são descartados;
- **espaços** são removidos das pontas e preservados no meio do texto. Caso exista uma sequência de espaços, eles são colapsados. 

Assim, qualquer entrada arbitrária é convertida para letras maiúsculas (`A–Z`) e espaços antes de ser cifrada ou decifrada.

## Tarefa 2: Força Bruta em César

### Lista de palavras muito comuns em português

In [1041]:
common_words = ["E", "DE", "DA", "QUE", "O", "A"]

### Escore simples de palavras comuns

In [1042]:
def score_text(text):
    """
    Calcula um escore simples baseado em palavras comuns.
    Quanto mais palavras comuns aparecem, maior o escore.

    Parâmetros:
    - text: string de entrada

    Retorno:
    - escore (int)
    """
    words = text.split()
    score = 0
    for w in words:
        if w in common_words:
            score += 1
    return score

### Frequências de letras em português (sem acentos) - porcentagem aproximada

In [1043]:
pt_freq = {
    'A': 14.63, 'B': 1.04, 'C': 3.88, 'D': 4.99, 'E': 12.57,
    'F': 1.02, 'G': 1.30, 'H': 1.28, 'I': 6.18, 'J': 0.40,
    'K': 0.02, 'L': 2.78, 'M': 4.74, 'N': 5.05, 'O': 10.73,
    'P': 2.52, 'Q': 1.20, 'R': 6.53, 'S': 7.81, 'T': 4.34,
    'U': 4.63, 'V': 1.67, 'W': 0.01, 'X': 0.21, 'Y': 0.01,
    'Z': 0.47
}

### Escore simples de frequência

In [1044]:
def freq_score(text):
    """
    Calcula a distância entre as frequências do texto e as do português.
    Quanto MENOR, melhor.

    Parâmetros:
    - text: string de entrada

    Retorno:
    - escore de frequência (float)
    """
    # Normaliza entrada (remove espaços)
    text = text.replace(" ", "")
    if not text:
        return float('inf')
    
    # Conta letras
    counts = Counter(text)
    total = sum(counts.values())
    
    # Frequência relativa em %
    freq = {ch: (counts.get(ch, 0) / total) * 100 for ch in pt_freq.keys()}
    
    # Soma dos quadrados das diferenças
    dist = sum((freq[ch] - pt_freq[ch])**2 for ch in pt_freq.keys())
    return dist

### Combina escores de palavras comuns e de frequência

In [1045]:
def combined_score(text):
    """
    Combina score de palavras comuns e proximidade do histograma.
    Palavras comuns -> bônus positivo
    Frequência -> penalidade (distância baixa é melhor)

    Parâmetros:
    - text: string de entrada

    Retorno:
    - escore combinado (float)
    """
    return score_text(text) - 0.01 * freq_score(text)

### Texto cifrado

In [1046]:
ciphertext = "SXBN N DV XCRVX NBCDMJWCN MJ DOVP"

### Hipóteses

In [1047]:
candidates = []
for k in range(1, 26):
    plain = cesar_decrypt(ciphertext, k)
    s_text = score_text(plain)
    s_freq = freq_score(plain)
    s_combined = combined_score(plain)
    candidates.append((s_text, s_freq, s_combined, k, plain))

### Hipóteses ranqueadas

#### Ranqueado por palavras comuns

In [1048]:
# ordenar por escore (desc), depois por k
candidates.sort(key=lambda x: (-x[0], x[3]))

i = 1
for s_text, s_freq, s_combined, k, plain in candidates:
    print(f"{i:2d}º | k = {k:2d} | score = {s_text} | {plain}")
    i += 1

 1º | k =  9 | score = 2 | JOSE E UM OTIMO ESTUDANTE DA UFMG
 2º | k = 13 | score = 1 | FKOA A QI KPEIK AOPQZWJPA ZW QBIC
 3º | k = 25 | score = 1 | TYCO O EW YDSWY OCDENKXDO NK EPWQ
 4º | k =  1 | score = 0 | RWAM M CU WBQUW MABCLIVBM LI CNUO
 5º | k =  2 | score = 0 | QVZL L BT VAPTV LZABKHUAL KH BMTN
 6º | k =  3 | score = 0 | PUYK K AS UZOSU KYZAJGTZK JG ALSM
 7º | k =  4 | score = 0 | OTXJ J ZR TYNRT JXYZIFSYJ IF ZKRL
 8º | k =  5 | score = 0 | NSWI I YQ SXMQS IWXYHERXI HE YJQK
 9º | k =  6 | score = 0 | MRVH H XP RWLPR HVWXGDQWH GD XIPJ
10º | k =  7 | score = 0 | LQUG G WO QVKOQ GUVWFCPVG FC WHOI
11º | k =  8 | score = 0 | KPTF F VN PUJNP FTUVEBOUF EB VGNH
12º | k = 10 | score = 0 | INRD D TL NSHLN DRSTCZMSD CZ TELF
13º | k = 11 | score = 0 | HMQC C SK MRGKM CQRSBYLRC BY SDKE
14º | k = 12 | score = 0 | GLPB B RJ LQFJL BPQRAXKQB AX RCJD
15º | k = 14 | score = 0 | EJNZ Z PH JODHJ ZNOPYVIOZ YV PAHB
16º | k = 15 | score = 0 | DIMY Y OG INCGI YMNOXUHNY XU OZGA
17º | k = 16 | score = 0

#### Ranqueado por frequência

In [1049]:
# ordenar por escore (desc), depois por k
candidates.sort(key=lambda x: (x[1], x[3]))

i = 1
for s_text, s_freq, s_combined, k, plain in candidates:
    print(f"{i:2d}º | k = {k:2d} | score = {s_freq} | {plain}")
    i += 1

 1º | k =  9 | score = 302.62769835390935 | JOSE E UM OTIMO ESTUDANTE DA UFMG
 2º | k = 21 | score = 568.6276983539095 | XCGS S IA CHWAC SGHIROBHS RO ITAU
 3º | k = 25 | score = 767.5906613168725 | TYCO O EW YDSWY OCDENKXDO NK EPWQ
 4º | k = 13 | score = 767.8128835390946 | FKOA A QI KPEIK AOPQZWJPA ZW QBIC
 5º | k = 10 | score = 783.8128835390949 | INRD D TL NSHLN DRSTCZMSD CZ TELF
 6º | k =  1 | score = 838.99806872428 | RWAM M CU WBQUW MABCLIVBM LI CNUO
 7º | k = 23 | score = 865.294365020576 | VAEQ Q GY AFUYA QEFGPMZFQ PM GRYS
 8º | k = 15 | score = 881.7388094650206 | DIMY Y OG INCGI YMNOXUHNY XU OZGA
 9º | k =  3 | score = 903.9610316872429 | PUYK K AS UZOSU KYZAJGTZK JG ALSM
10º | k = 24 | score = 938.2573279835392 | UZDP P FX ZETXZ PDEFOLYEP OL FQXR
11º | k = 11 | score = 985.3684390946503 | HMQC C SK MRGKM CQRSBYLRC BY SDKE
12º | k = 17 | score = 986.2573279835393 | BGKW W ME GLAEG WKLMVSFLW VS MXEY
13º | k = 19 | score = 989.220290946502 | ZEIU U KC EJYCE UIJKTQDJU TQ KVCW
14

#### Ranqueado pela combinação de palavras comuns e frequência

In [1050]:
# ordenar por escore (desc), depois por k
candidates.sort(key=lambda x: (-x[2], x[3]))

i = 1
for s_text, s_freq, s_combined, k, plain in candidates:
    print(f"{i:2d}º | k = {k:2d} | score = {s_combined} | {plain}")
    i += 1

 1º | k =  9 | score = -1.0262769835390935 | JOSE E UM OTIMO ESTUDANTE DA UFMG
 2º | k = 21 | score = -5.686276983539095 | XCGS S IA CHWAC SGHIROBHS RO ITAU
 3º | k = 25 | score = -6.675906613168725 | TYCO O EW YDSWY OCDENKXDO NK EPWQ
 4º | k = 13 | score = -6.678128835390947 | FKOA A QI KPEIK AOPQZWJPA ZW QBIC
 5º | k = 10 | score = -7.838128835390949 | INRD D TL NSHLN DRSTCZMSD CZ TELF
 6º | k =  1 | score = -8.389980687242799 | RWAM M CU WBQUW MABCLIVBM LI CNUO
 7º | k = 23 | score = -8.65294365020576 | VAEQ Q GY AFUYA QEFGPMZFQ PM GRYS
 8º | k = 15 | score = -8.817388094650207 | DIMY Y OG INCGI YMNOXUHNY XU OZGA
 9º | k =  3 | score = -9.039610316872428 | PUYK K AS UZOSU KYZAJGTZK JG ALSM
10º | k = 24 | score = -9.382573279835393 | UZDP P FX ZETXZ PDEFOLYEP OL FQXR
11º | k = 11 | score = -9.853684390946503 | HMQC C SK MRGKM CQRSBYLRC BY SDKE
12º | k = 17 | score = -9.862573279835393 | BGKW W ME GLAEG WKLMVSFLW VS MXEY
13º | k = 19 | score = -9.89220290946502 | ZEIU U KC EJYCE UIJKT

### 5 Melhores hipóteses

#### Palavras comuns

In [1051]:
candidates.sort(key=lambda x: (-x[0], x[3]))

i = 1
for s_text, s_freq, s_combined, k, plain in candidates[:5]:
    print(f"{i:2d}º | k = {k:2d} | score = {s_text} | {plain}")
    i += 1

 1º | k =  9 | score = 2 | JOSE E UM OTIMO ESTUDANTE DA UFMG
 2º | k = 13 | score = 1 | FKOA A QI KPEIK AOPQZWJPA ZW QBIC
 3º | k = 25 | score = 1 | TYCO O EW YDSWY OCDENKXDO NK EPWQ
 4º | k =  1 | score = 0 | RWAM M CU WBQUW MABCLIVBM LI CNUO
 5º | k =  2 | score = 0 | QVZL L BT VAPTV LZABKHUAL KH BMTN


#### Frequência

In [1052]:
candidates.sort(key=lambda x: (x[1], x[3]))

i = 1
for s_text, s_freq, s_combined, k, plain in candidates[:5]:
    print(f"{i:2d}º | k = {k:2d} | score = {s_freq} | {plain}")
    i += 1

 1º | k =  9 | score = 302.62769835390935 | JOSE E UM OTIMO ESTUDANTE DA UFMG
 2º | k = 21 | score = 568.6276983539095 | XCGS S IA CHWAC SGHIROBHS RO ITAU
 3º | k = 25 | score = 767.5906613168725 | TYCO O EW YDSWY OCDENKXDO NK EPWQ
 4º | k = 13 | score = 767.8128835390946 | FKOA A QI KPEIK AOPQZWJPA ZW QBIC
 5º | k = 10 | score = 783.8128835390949 | INRD D TL NSHLN DRSTCZMSD CZ TELF


#### Combinação de palavras comuns e frequência

In [1053]:
candidates.sort(key=lambda x: (-x[2], x[3]))

i = 1
for s_text, s_freq, s_combined, k, plain in candidates[:5]:
    print(f"{i:2d}º | k = {k:2d} | score = {s_combined} | {plain}")
    i += 1

 1º | k =  9 | score = -1.0262769835390935 | JOSE E UM OTIMO ESTUDANTE DA UFMG
 2º | k = 21 | score = -5.686276983539095 | XCGS S IA CHWAC SGHIROBHS RO ITAU
 3º | k = 25 | score = -6.675906613168725 | TYCO O EW YDSWY OCDENKXDO NK EPWQ
 4º | k = 13 | score = -6.678128835390947 | FKOA A QI KPEIK AOPQZWJPA ZW QBIC
 5º | k = 10 | score = -7.838128835390949 | INRD D TL NSHLN DRSTCZMSD CZ TELF


### k e plaintext final

In [1054]:
best = candidates[0]
print(f"k = {best[3]} | plaintext: {best[4]}")

k = 9 | plaintext: JOSE E UM OTIMO ESTUDANTE DA UFMG


## Tarefa 3: Análise de Frequência em César

### Bigramas prováveis

In [1055]:
bigrams = ["DE", "AS", "OS", "DO"]

In [1056]:
def count_bigrams(text, bigrams):
    """
    Conta ocorrências dos bigramas especificados no texto.
    
    Parâmetros:
    - text: string de entrada

    Retorno:
    - dicionário com contagem de cada bigrama
    """
    text = normalize(text)
    counts = Counter([text[i:i+2] for i in range(len(text)-1)])
    return {bg: counts.get(bg, 0) for bg in bigrams}

### Texto cifrado

In [1057]:
ciphertext = "RKRHLV UV UZTZFERIZF V VWZTRQ TFEKIR JVEYRJ WIRTRJ"

### Histograma de frequência das letras do ciphertext (letras e porcentagem)

In [1058]:
# Normaliza sem espaços
ct_norm = normalize(ciphertext, keep_spaces=False)

# Conta letras
counts = Counter(ct_norm)
total_letters = sum(counts.values())

# Tabela de frequências
freq_table = []
for ch, cnt in counts.most_common():
    pct = (cnt / total_letters) * 100
    freq_table.append((ch, cnt, pct))
    print(f"{ch}: {cnt} ({pct:.2f}%)")

R: 8 (18.60%)
V: 5 (11.63%)
Z: 4 (9.30%)
T: 4 (9.30%)
F: 3 (6.98%)
E: 3 (6.98%)
I: 3 (6.98%)
J: 3 (6.98%)
K: 2 (4.65%)
U: 2 (4.65%)
W: 2 (4.65%)
H: 1 (2.33%)
L: 1 (2.33%)
Q: 1 (2.33%)
Y: 1 (2.33%)


### Estimativa com letra mais frequente do texto correspondendo a **E** - Segunda letra mais frequente do português

#### Estimativa de k 

In [1059]:
most_common_cipher_letter = counts.most_common(1)[0][0]
print("\nLetra mais frequente no ciphertext:", most_common_cipher_letter)

assumed_plain = 'E'
k_est = (ord(most_common_cipher_letter) - ord(assumed_plain)) % 26
print(f"Assumindo que a letra mais frequente '{most_common_cipher_letter}' corresponde a 'E' no plaintext. Estimativa de k =", k_est)



Letra mais frequente no ciphertext: R
Assumindo que a letra mais frequente 'R' corresponde a 'E' no plaintext. Estimativa de k = 13


#### Plaintext estimado

In [1060]:
plaintext_est = cesar_decrypt(ciphertext, k_est)

print(f"Plaintext estimado com k = {k_est}:\n")
print(plaintext_est)

Plaintext estimado com k = 13:

EXEUYI HI HMGMSREVMS I IJMGED GSRXVE WIRLEW JVEGEW


#### Contagem de bigramas prováveis

In [1061]:
bigram_counts = count_bigrams(plaintext_est, bigrams)

print("Contagem dos bigramas candidatos (DE, AS, OS, DO):")
for bg, c in bigram_counts.items():
    print(f"{bg}: {c}")

Contagem dos bigramas candidatos (DE, AS, OS, DO):
DE: 0
AS: 0
OS: 0
DO: 0


### Estimativa com letra mais frequente do texto correspondendo a **A** - Letra mais frequente do português

#### Estimativa de k 

In [1062]:
most_common_cipher_letter = counts.most_common(1)[0][0]
print("\nLetra mais frequente no ciphertext:", most_common_cipher_letter)

assumed_plain = 'A'
k_est = (ord(most_common_cipher_letter) - ord(assumed_plain)) % 26
print(f"Assumindo que a letra mais frequente '{most_common_cipher_letter}' corresponde a 'A' no plaintext. Estimativa de k =", k_est)



Letra mais frequente no ciphertext: R
Assumindo que a letra mais frequente 'R' corresponde a 'A' no plaintext. Estimativa de k = 17


#### Plaintext estimado

In [1063]:
plaintext_est = cesar_decrypt(ciphertext, k_est)

print(f"Plaintext estimado com k = {k_est}:\n")
print(plaintext_est)

Plaintext estimado com k = 17:

ATAQUE DE DICIONARIO E EFICAZ CONTRA SENHAS FRACAS


#### Contagem de bigramas prováveis

In [1064]:
bigram_counts = count_bigrams(plaintext_est, bigrams)

print("Contagem dos bigramas candidatos (DE, AS, OS, DO):")
for bg, c in bigram_counts.items():
    print(f"{bg}: {c}")

Contagem dos bigramas candidatos (DE, AS, OS, DO):
DE: 1
AS: 2
OS: 0
DO: 0


## Tarefa 4: Implementação da Cifra de Vigenère

### Cifra de Vigenère - Cifrar

In [1065]:
def vigenere_cifrar(text, key):
    """
    Cifra de Vigenère: desloca cada letra de acordo com a chave (cíclica).
    Espaços são preservados.

    Parâmetros:
    - text: string de entrada (plaintext)
    - key: string com a chave (apenas letras)

    Retorno:
    - texto cifrado. Mantém espaços, apenas letras são deslocadas.
    """
    text = normalize(text)
    key = normalize(key, keep_spaces=False)  # chave só com letras
    result = []
    j = 0  # índice para a chave

    for char in text:
        if char == " ":
            result.append(" ")
        else:
            displaced = ord(key[j % len(key)]) - ord("A")
            new = (ord(char) - ord("A") + displaced) % 26 + ord("A")
            result.append(chr(new))
            j += 1
    return "".join(result)

### Cifra de Vigenère - Decifrar

In [1066]:
def vigenere_decifrar(text, key):
    """
    Decifrar a Cifra de Vigenère: desfaz o deslocamento da chave.

    Parâmetros:
    - text: string de entrada (plaintext)
    - key: string com a chave (apenas letras)

    Retorno:
    - texto cifrado. Mantém espaços, apenas letras são deslocadas.
    """
    text = normalize(text)
    key = normalize(key, keep_spaces=False)  # chave só com letras
    result = []
    j = 0

    for char in text:
        if char == " ":
            result.append(" ")
        else:
            displaced = ord(key[j % len(key)]) - ord("A")
            new = (ord(char) - ord("A") - displaced) % 26 + ord("A")
            result.append(chr(new))
            j += 1
    return "".join(result)

### Validação

In [1067]:
original_text = "CIFRA DE VIGENERE E UM POUCO MAIS FORTE"
key = "SEGURO"

In [1068]:
encrypt = vigenere_cifrar(original_text, key)
decrypt = vigenere_decifrar(encrypt, key)

In [1069]:
print("Texto original:", original_text)
print("Cifrado:", encrypt)
print("Decifrado:", decrypt)

Texto original: CIFRA DE VIGENERE E UM POUCO MAIS FORTE
Cifrado: UMLLR RW ZOAVBWVK Y LA HSAWF ASMY ZFFLI
Decifrado: CIFRA DE VIGENERE E UM POUCO MAIS FORTE


### Resultado

Para validar a implementação, a mensagem `CIFRA DE VIGENERE E UM POUCO MAIS FORTE` foi cifrada com a chave `SEGURO`. Obtendo como resultado `UMLLR RW ZOAVBWVK Y LA HSAWF ASMY ZFFLI`.

Em seguida, o texto foi decifrado com a mesma chave `SEGURO`, revelando novamente a mensagem original: `CIFRA DE VIGENERE E UM POUCO MAIS FORTE`.

#### Uso cíclico da chave

Na implementação da Cifra de Vigenère, foi utilizada uma chave cíclica sobre o alfabeto `A–Z`, preservando espaços e normalizando o texto (maiúsculas, sem acentos).

A chave fornecida foi **SEGURO**. 

Durante o processo de cifragem, a chave é repetida ciclicamente até cobrir todo o texto:

    TEXTO: CIFRA DE VIGENERE E UM POUCO MAIS FORTE
    CHAVE: SEGUR OS EGUROSEG U RO SEGUR OSEG UROSE

Cada letra do texto é deslocada conforme o valor da letra correspondente da chave.  
Por exemplo:
- C (texto) + S (chave) → U  
- I (texto) + E (chave) → M  
- F (texto) + G (chave) → L  

#### Tratamento de espaços

Os espaços não são alterados pelo algoritmo. Eles são simplesmente preservados na mesma posição, garantindo que a mensagem cifrada e decifrada mantenha o formato do texto original.

## Tarefa 5: Quebra de Vigenère (IC / Kasiski + César por coluna)

### Funções para a quebra de cifra

In [1070]:
# Constantes e variáveis de apoio
ALPHABET = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
IDX = {letra: i for i, letra in enumerate(ALPHABET)}

In [1071]:
# Converte as porcentagens de pt_freq para frações decimais
FREQ_PT = {k: v / 100.0 for k, v in pt_freq.items()}

In [1072]:
def partition_columns(text: str, N: int) -> list[str]:
    """
    Divide o texto em N colunas, distribuindo os caracteres em round-robin.
    
    Parâmetros:
    - text: string de entrada sem espaços
    - N: número de colunas

    Retorno:
    - lista com N strings, cada uma representando uma coluna
    """
    columns = ['' for _ in range(N)]
    for i, char in enumerate(text):
        columns[i % N] += char
    return columns

In [1073]:
def coincidence_index(column: str) -> float:
    """
    Calcula o Índice de Coincidência de uma string (A-Z).

    Parâmetros:
    - column: string de entrada (apenas letras A-Z)

    Retorno:
    - índice de coincidência (float)
    """
    n = len(column)
    if n < 2:
        return 0.0
    cont = Counter(column)
    numerator = sum(f*(f-1) for f in cont.values())
    denominator = n*(n-1)
    return numerator/denominator

In [1074]:
def chi_squared(text: str) -> list[float]:
    """
    Calcula o qui-quadrado para cada possível deslocamento (0-25) do texto.

    Parâmetros:
    - text: string de entrada (apenas letras A-Z)

    Retorno:
    - lista chi2[k] = qui-quadrado do texto deslocado por k 
    """
    n = len(text)
    if n == 0:
        return [float('inf')]*26
    cont = Counter(text)
    chi = []
    for k in range(26):
        sum = 0.0
        for char in ALPHABET:
            exp = FREQ_PT[char]*n
            obs = cont[ALPHABET[(IDX[char]+k)%26]]
            if exp > 0:
                d = obs - exp
                sum = sum + (d**2)/exp
        chi.append(sum)
    return chi

In [1075]:
def reassemble_text(columns: list[str], N: int, original_text: str) -> str:
    """
    Reconstroi o texto original a partir das colunas e do texto original com espaços.

    Parâmetros:
    - columns: lista de strings, cada uma representando uma coluna
    - N: número de colunas
    - original_text: texto original com espaços     

    Retorno:
    - texto reconstituído, com espaços na mesma posição do original
    """
    out = []
    idxs = [0]*N
    j = 0
    for char in original_text:
        if char == ' ':
            out.append(' ')
        else:
            col = j % N
            out.append(columns[col][idxs[col]])
            idxs[col] += 1
            j += 1
    return ''.join(out)

In [1076]:
def calculate_score(pt: str) -> int:
    """
    Calcula um escore simples baseado em bigramas e palavras comuns.

    Parâmetros:
    - pt: string de entrada

    Retorno:
    - escore (int)
    """
    bigrams = ["AS","DA","DE","DO","EM","OS","QUE"]
    words = ["A","AS","COM","DA","DE","DO","E","EM","NA","NO","O","OS","POR","QUE","UM","UMA"]
    tokens = pt.split()
    return sum(pt.count(bg) for bg in bigrams) + sum(tokens.count(w) for w in words)

In [1077]:
def kasiski(text: str, min_len=2, max_len=4, kmin=2, kmax=16, top=5) -> list[int]:
    """
    Kasiski simples: repetição 2..4 letras -> distâncias -> múltiplos de m.
    
    Parâmetros:
    - text: string de entrada (apenas letras A-Z)
    - min_len: tamanho mínimo do n-grama (default=2)
    - max_len: tamanho máximo do n-grama (default=4)
    - kmin: tamanho mínimo da chave (default=2)
    - kmax: tamanho máximo da chave (default=16)
    - top: número máximo de candidatos a retornar (default=5)

    Retorno:
    - lista com os tamanhos de chave mais prováveis (inteiros)
    """
    # Encontrar as posições de todos os n-gramas
    pos = defaultdict(list)
    n = len(text)
    for length in range(min_len, max_len+1):
        for i in range(n-length+1):
            sub = text[i:i+length]
            pos[sub].append(i)

    # Calcular as distâncias entre as ocorrências de n-gramas repetidos     
    distances = (
        pos_list[i + 1] - pos_list[i]
        for pos_list in pos.values()
        if len(pos_list) > 1
        for i in range(len(pos_list) - 1)
    )
    
    # Encontrar todos os fatores de todas as distâncias e contá-los
    factor_counts = Counter(
        key_len
        for dist in distances
        for key_len in range(kmin, kmax + 1)
        if dist % key_len == 0
    )
    
    # Retornar os fatores mais comuns, que são os comprimentos de chave mais prováveis
    most_likely_keys = [key for key, count in factor_counts.most_common(top)]
    
    return most_likely_keys

In [1078]:
def top_shifts(col: str, top_k: int = 3):
    """
    Top-k deslocamentos por coluna (menor qui-quadrado)

    Parâmetros:
    - col: string de entrada (apenas letras A-Z)
    - top_k: número de deslocamentos a retornar (default=3)     

    Retorno:
    - lista de tuplas (k, chi2) com os top-k deslocamentos e seus qui-quadrados
    """
    chi = chi_squared(col)
    idxs = sorted(range(26), key=lambda k: chi[k])[:top_k]
    return [(k, chi[k]) for k in idxs]

In [1079]:
def find_key_length_candidates(clean_text: str, max_key_len: int, top_n: int) -> tuple[list[int], dict[int, float]]:
    """
    Encontra prováveis comprimentos de chave usando Índice de Coincidência e Kasiski.

    Parâmetros:
    - clean_text: Texto limpo a ser analisado.
    - max_key_len: Comprimento máximo da chave a ser considerado.
    - top_n: Número de melhores candidatos a serem retornados.

    Retorno:
    - Uma tupla contendo:
        - Uma lista dos melhores comprimentos de chave.
        - Um dicionário com o Índice de Coincidência médio para cada comprimento de chave.
    """
    # Etapa 1: Análise pelo Índice de Coincidência (IC)
    ic_by_length = {}
    limit = min(max_key_len, len(clean_text))
    for key_len in range(1, limit + 1):
        cols = partition_columns(clean_text, key_len)
        ics = [coincidence_index(c) for c in cols if c]
        ic_by_length[key_len] = sum(ics) / len(ics) if ics else 0.0

    # Seleciona os melhores candidatos pelo IC
    ic_candidates = sorted(ic_by_length, key=ic_by_length.get, reverse=True)[:top_n]

    # Etapa 2: Análise por Kasiski
    kasiski_candidates = kasiski(clean_text, kmin=2, kmax=max_key_len, top=top_n)

    # Combina e retorna os candidatos únicos
    combined_candidates = sorted(list(set(ic_candidates + kasiski_candidates)))
    
    return combined_candidates, ic_by_length

In [1080]:
def prune_combinations_with_beam_search(shift_options_per_col: list[list[int]], cols: list[str], beam_limit: int) -> list[tuple[int, ...]]:
    """
    Gera combinações de deslocamentos e as poda se excederem o limite (beam_limit).

    Parâmetros:
    - shift_options_per_col: Lista de listas com opções de deslocamento para cada coluna.
    - cols: Lista de colunas do texto cifrado.
    - beam_limit: Limite de combinações a serem mantidas.

    Retorno:
    - Uma lista de combinações de deslocamento podadas.
    """
    all_combinations = list(product(*shift_options_per_col))

    # Se o número de combinações for muito grande, aplicamos a poda
    if len(all_combinations) <= beam_limit:
        return all_combinations

    # Calcula o qui-quadrado para cada coluna uma única vez
    chi_squared_per_col = [chi_squared(col) for col in cols]
    
    # Pontua cada combinação pela soma dos qui-quadrados
    scored_combinations = []
    num_cols = len(cols)
    for combo in all_combinations:
        # A pontuação é a soma do qui-quadrado de cada deslocamento escolhido
        score = sum(chi_squared_per_col[i][combo[i]] for i in range(num_cols))
        scored_combinations.append((score, combo))
    
    # Mantém apenas as melhores (menor qui-quadrado)
    scored_combinations.sort(key=lambda x: x[0])
    
    return [combo for score, combo in scored_combinations[:beam_limit]]

In [1081]:
def evaluate_best_key_for_length( key_combinations: list[tuple[int, ...]], cols: list[str], key_length: int, normalized_text: str) -> dict | None:
    """
    Testa todas as combinações de chave, decifra e pontua para encontrar a melhor.

    Parâmetros:
    - key_combinations: Lista de combinações de deslocamento para testar.
    - cols: Lista de colunas do texto cifrado.
    - key_length: Comprimento da chave.
    - normalized_text: Texto normalizado para referência.

    Retorno:
    - Um dicionário com os melhores resultados ou None se não houver resultados válidos.
    """
    best_result = {"score": -1, "key": None, "key_shifts": None, "plaintext": None}

    for key_shifts in key_combinations:
        # Decifra as colunas com os deslocamentos atuais
        decrypted_cols = [
            ''.join(ALPHABET[(IDX[char] - key_shifts[i]) % 26] for char in cols[i])
            for i in range(key_length)
        ]
        
        # Remonta o texto plano, preservando espaços e pontuação original
        plaintext = reassemble_text(decrypted_cols, key_length, normalized_text)
        score = calculate_score(plaintext)

        if score > best_result["score"]:
            best_result = {
                "score": score,
                "key": ''.join(ALPHABET[k] for k in key_shifts),
                "key_shifts": list(key_shifts),
                "plaintext": plaintext
            }
            
    return best_result if best_result["key"] else None

In [1082]:
def vigenere_solver(ciphertext: str, max_key_len: int = 12, top_n_M: int = 3, top_k: int = 3, beam_limit: int = 2048):
    """
    Decifra um texto com a Cifra de Vigenère usando uma abordagem híbrida.
    
    Combina o Exame de Kasiski e o Índice de Coincidência para encontrar o tamanho da chave,
    e usa uma busca em feixe (beam search) para encontrar a chave de decifragem.

    Parâmetros:
    - ciphertext: O texto cifrado a ser decifrado.
    - max_key_len: O comprimento máximo da chave a ser considerado.
    - top_n_M: O número de melhores candidatos a tamanhos de chave a serem considerados.
    - top_k: O número de melhores deslocamentos a serem considerados para cada coluna.
    - beam_limit: O limite de combinações a serem mantidas durante a busca.

    Retorno:
    - Um dicionário com os resultados da decifragem, incluindo candidatos a tamanhos de chave, Índice de Coincidência e melhores resultados.
    """
    # Etapa 1: Preparar o texto
    text_norm = normalize(ciphertext)
    text_clean = normalize(ciphertext, keep_spaces=False)
    if not text_clean:
        return {"error": "Texto vazio após normalização."}

    # Etapa 2: Encontrar prováveis tamanhos de chave (M)
    key_len_candidates, ic_by_m = find_key_length_candidates(text_clean, max_key_len, top_n_M)
    if not key_len_candidates:
        return {"error": "Nenhum candidato a tamanho de chave foi encontrado."}

    # Etapa 3: Iterar sobre os tamanhos de chave e encontrar a melhor chave/decifragem
    all_results = []
    for m in key_len_candidates:
        cols = partition_columns(text_clean, m)
        
        # Encontra os melhores 'k' deslocamentos para cada coluna
        top_shifts_per_col = [
            [shift for shift, score in top_shifts(col, top_k=top_k)] 
            for col in cols
        ]
        
        # Gera e poda as combinações de chave
        key_combinations = prune_combinations_with_beam_search(top_shifts_per_col, cols, beam_limit)
        
        # Avalia as combinações e encontra a melhor para este 'm'
        best_for_m = evaluate_best_key_for_length(key_combinations, cols, m, text_norm)

        if best_for_m:
            all_results.append({
                "M": m,
                "avg_ic": ic_by_m.get(m, 0.0),
                **best_for_m  # Adiciona score, key, plaintext, etc.
            })

    # Etapa 4: Classificar e formatar o resultado final
    all_results.sort(key=lambda d: (d["score"], d["avg_ic"]), reverse=True)
    
    return {
        "key_len_candidates": key_len_candidates,
        "ic_by_m": ic_by_m,
        "best": all_results[0] if all_results else None,
        "top_results": all_results[:5]
    }

### Execução da quebra de cifra

In [1083]:
ciphertext = "ECXPFI WO QQYFO LX JWOXBSZX"

In [1084]:
result = vigenere_solver(ciphertext)

best = result["best"]
if best:
    print(f"Tamanho da chave: {best['M']}")
    print(f"IC médio: {best['avg_ic']}")
    print(f"Escore: {best['score']}")
    print(f"Chave: {best['key']}")
    print(f"Texto decifrado: {best['plaintext']}")
else:
    print("Não foi possível encontrar uma solução.")

Tamanho da chave: 4
IC médio: 0.1
Escore: 5
Chave: OITO
Texto decifrado: QUEBRA DA CIFRA DE VIGENERE
